# 🫁 Chest X-Ray Pneumonia Classifier — MLOps Notebook

**Assignment:** MLOps Summative — End-to-End ML Pipeline  
**Dataset:** [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia)  
**Model:** MobileNetV2 Transfer Learning (Binary Classification: NORMAL vs PNEUMONIA)  

---

## Pipeline Overview

| Step | Description |
|------|-------------|
| 1 | Install dependencies |
| 2 | Load & explore the dataset |
| 3 | Preprocess images (resize, normalize, augment) |
| 4 | Exploratory Data Analysis (EDA) |
| 5 | Build MobileNetV2 model |
| 6 | Train — Phase 1 (frozen base) |
| 7 | Fine-tune — Phase 2 (unfreeze top 20 layers) |
| 8 | Evaluate with 6 metrics |
| 9 | Save model & download |
| 10 | Prediction demo |


---
## Step 1 — Install Dependencies

We install all required libraries. TensorFlow comes pre-installed on Google Colab with GPU support, but we explicitly pin versions to ensure reproducibility.

In [ ]:
# Install / upgrade required packages
# (Most are already on Colab; this ensures the correct versions are present)
!pip install -q tensorflow==2.15.0 scikit-learn matplotlib seaborn Pillow numpy pandas

In [ ]:
import os
import io
import json
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.metrics import Precision, Recall, AUC
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc as sklearn_auc,
    f1_score,
    precision_score,
    recall_score,
)

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")
print("GPU devices:", tf.config.list_physical_devices('GPU'))

---
## Step 2 — Load the Dataset

We use the [Kaggle Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia) dataset. 
It contains **5,863 JPEG chest X-ray images** organised into two classes:

- **NORMAL** — healthy lung X-rays
- **PNEUMONIA** — X-rays showing either bacterial or viral pneumonia

**Option A (Recommended in Colab):** Use `tensorflow_datasets` — no Kaggle credentials needed.  
**Option B:** Upload dataset manually from Kaggle and unzip.

In [ ]:
# ── Option A: Load via tensorflow_datasets (no Kaggle login needed) ──────────
# This downloads ~1.2 GB to Colab's /root/tensorflow_datasets directory.

!pip install -q tensorflow-datasets
import tensorflow_datasets as tfds

# Load dataset — splits: 'train', 'test', 'validation'
# chest_xray_pneumonia labels: 0 = NORMAL, 1 = PNEUMONIA
(ds_train_raw, ds_val_raw, ds_test_raw), ds_info = tfds.load(
    'chest_xray_pneumonia',
    split=['train', 'validation', 'test'],
    with_info=True,
    as_supervised=True,  # returns (image, label) tuples
)

print(ds_info)
print(f"\nTrain samples: {len(ds_train_raw)}")
print(f"Validation samples: {len(ds_val_raw)}")
print(f"Test samples: {len(ds_test_raw)}")

CLASS_NAMES = ['NORMAL', 'PNEUMONIA']
IMG_SIZE = 224
BATCH_SIZE = 32

In [ ]:
# ── Option B: If you uploaded the Kaggle dataset manually ────────────────────
# Unzip and define paths — only run this block if NOT using Option A above.

# !unzip -q /content/chest-xray-pneumonia.zip -d /content/chest_xray

# TRAIN_DIR = '/content/chest_xray/chest_xray/train'
# TEST_DIR  = '/content/chest_xray/chest_xray/test'
# VAL_DIR   = '/content/chest_xray/chest_xray/val'

print("(Option B paths are commented out — using Option A with tensorflow_datasets)")

---
## Step 3 — Preprocessing

Preprocessing converts raw images into normalized tensors suitable for MobileNetV2:

1. **Resize** every image to 224×224 pixels (MobileNetV2's expected input size)
2. **Normalize** pixel values from `[0, 255]` → `[0.0, 1.0]`
3. **One-hot encode** labels for categorical crossentropy loss
4. **Data augmentation** on the training set only:
   - Random horizontal flip
   - Random rotation up to ±15°
   - Random zoom up to ±10%
   - Brightness jitter in `[0.8, 1.2]`

In [ ]:
# ── Preprocessing pipeline (for tensorflow_datasets path) ────────────────────

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 2

def preprocess_image(image, label):
    """
    Resize image to (IMG_SIZE, IMG_SIZE, 3) and normalise pixels to [0, 1].
    Cast label to one-hot encoding for categorical crossentropy.
    """
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])      # → (224, 224, 3)
    image = tf.cast(image, tf.float32) / 255.0                 # → [0.0, 1.0]
    label = tf.one_hot(label, depth=NUM_CLASSES)               # → [1,0] or [0,1]
    return image, label


def augment_image(image, label):
    """
    Apply random augmentations to TRAINING images only.
    Augmentation artificially enlarges the training distribution,
    reducing overfitting on the smaller NORMAL class.
    """
    image = tf.image.random_flip_left_right(image)                      # horizontal flip
    image = tf.image.random_brightness(image, max_delta=0.2)            # brightness ±20%
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)       # contrast jitter
    image = tf.clip_by_value(image, 0.0, 1.0)                           # keep in [0, 1]
    return image, label


AUTOTUNE = tf.data.AUTOTUNE

# Training pipeline: preprocess → augment → shuffle → batch → prefetch
train_ds = (
    ds_train_raw
    .map(preprocess_image, num_parallel_calls=AUTOTUNE)
    .map(augment_image,    num_parallel_calls=AUTOTUNE)
    .shuffle(buffer_size=1000, seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# Validation pipeline: preprocess only — NO augmentation
val_ds = (
    ds_val_raw
    .map(preprocess_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# Test pipeline: preprocess only — NO augmentation
test_ds = (
    ds_test_raw
    .map(preprocess_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print("Datasets prepared:")
print(f"  train_ds batches: {len(train_ds)}")
print(f"  val_ds batches:   {len(val_ds)}")
print(f"  test_ds batches:  {len(test_ds)}")

---
## Step 4 — Exploratory Data Analysis (EDA)

Before training, we explore the dataset to understand its structure:
1. Class distribution (are classes balanced?)
2. Sample images from each class
3. Pixel intensity distributions

Understanding the data first helps us choose the right model design and interpret results.

In [ ]:
# ── Feature 1: Class Distribution ────────────────────────────────────────────
# Count how many images belong to each class in the training set.

label_counts = {0: 0, 1: 0}  # 0=NORMAL, 1=PNEUMONIA
for _, label in ds_train_raw:
    label_counts[int(label)] += 1

normal_count    = label_counts[0]
pneumonia_count = label_counts[1]
total           = normal_count + pneumonia_count

print(f"Training set class distribution:")
print(f"  NORMAL:    {normal_count:,} images ({normal_count/total:.1%})")
print(f"  PNEUMONIA: {pneumonia_count:,} images ({pneumonia_count/total:.1%})")
print(f"  Imbalance ratio (PNEUMONIA:NORMAL): {pneumonia_count/normal_count:.2f}:1")

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
ax = axes[0]
bars = ax.bar(
    CLASS_NAMES,
    [normal_count, pneumonia_count],
    color=['#28a745', '#dc3545'],
    edgecolor='white',
    linewidth=1.5,
    width=0.5,
)
for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 30,
        f'{int(bar.get_height()):,}',
        ha='center', va='bottom', fontsize=13, fontweight='bold'
    )
ax.set_title('Training Set Class Distribution', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Images', fontsize=12)
ax.set_ylim(0, max(normal_count, pneumonia_count) * 1.15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Pie chart
ax2 = axes[1]
ax2.pie(
    [normal_count, pneumonia_count],
    labels=CLASS_NAMES,
    colors=['#28a745', '#dc3545'],
    autopct='%1.1f%%',
    startangle=90,
    textprops={'fontsize': 12},
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
)
ax2.set_title('Class Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n📊 Interpretation:")
print(f"The dataset is IMBALANCED — PNEUMONIA has {pneumonia_count/normal_count:.1f}x more images than NORMAL.")
print("This can bias the model toward the majority class (PNEUMONIA).")
print("Mitigation: Use data augmentation, monitor F1/Recall, not just accuracy.")

In [ ]:
# ── Feature 2: Sample Images from each class ─────────────────────────────────
# Visualise a few training examples to understand what the model is learning from.

normal_imgs    = []
pneumonia_imgs = []

for image, label in ds_train_raw.take(200):
    img_np = image.numpy()
    if int(label) == 0 and len(normal_imgs) < 5:
        normal_imgs.append(img_np)
    elif int(label) == 1 and len(pneumonia_imgs) < 5:
        pneumonia_imgs.append(img_np)
    if len(normal_imgs) == 5 and len(pneumonia_imgs) == 5:
        break

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
fig.suptitle('Sample Chest X-Ray Images', fontsize=16, fontweight='bold', y=1.02)

for i, img in enumerate(normal_imgs):
    axes[0, i].imshow(img, cmap='bone')
    axes[0, i].set_title(f'NORMAL #{i+1}', color='#28a745', fontweight='bold')
    axes[0, i].axis('off')

for i, img in enumerate(pneumonia_imgs):
    axes[1, i].imshow(img, cmap='bone')
    axes[1, i].set_title(f'PNEUMONIA #{i+1}', color='#dc3545', fontweight='bold')
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Interpretation:")
print("NORMAL X-rays show clear, dark lung fields with well-defined boundaries.")
print("PNEUMONIA X-rays show cloudy/hazy areas (infiltrates) in the lung fields,")
print("particularly in bacterial pneumonia which appears as dense white patches.")
print("Viral pneumonia tends to show more diffuse, bilateral interstitial patterns.")

In [ ]:
# ── Feature 3: Pixel Intensity Analysis ──────────────────────────────────────
# Compare the average pixel brightness between NORMAL and PNEUMONIA X-rays.
# PNEUMONIA lungs are expected to be brighter (higher intensity) due to
# fluid/consolidation showing as white regions on X-rays.

normal_intensities    = []
pneumonia_intensities = []

for image, label in ds_train_raw.take(500):
    # Convert to float [0,1] and compute mean pixel brightness
    img_float = tf.cast(image, tf.float32) / 255.0
    mean_brightness = float(tf.reduce_mean(img_float).numpy())
    if int(label) == 0:
        normal_intensities.append(mean_brightness)
    else:
        pneumonia_intensities.append(mean_brightness)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(normal_intensities,    bins=30, alpha=0.7, color='#28a745', label='NORMAL',    edgecolor='white')
axes[0].hist(pneumonia_intensities, bins=30, alpha=0.7, color='#dc3545', label='PNEUMONIA', edgecolor='white')
axes[0].set_title('Pixel Intensity Distribution by Class', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Mean Pixel Intensity (0=black, 1=white)')
axes[0].set_ylabel('Number of Images')
axes[0].legend()
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Box plot
axes[1].boxplot(
    [normal_intensities, pneumonia_intensities],
    labels=['NORMAL', 'PNEUMONIA'],
    patch_artist=True,
    boxprops=dict(facecolor='lightblue', color='navy'),
    medianprops=dict(color='red', linewidth=2),
)
axes[1].set_title('Pixel Intensity Box Plot', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Mean Pixel Intensity')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('pixel_intensity.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Interpretation:")
print(f"Mean brightness — NORMAL:    {sum(normal_intensities)/len(normal_intensities):.4f}")
print(f"Mean brightness — PNEUMONIA: {sum(pneumonia_intensities)/len(pneumonia_intensities):.4f}")
print()
print("PNEUMONIA X-rays tend to have HIGHER mean pixel intensity than NORMAL X-rays.")
print("This is because infected lung tissue (consolidation, fluid) appears WHITE on X-ray,")
print("increasing the overall brightness of the image. Healthy lungs appear darker because")
print("air-filled tissue is radio-transparent (X-rays pass through easily).")
print("This feature alone provides a weak signal — which is why deep CNNs are needed.")

---
## Step 5 — Build the MobileNetV2 Model

**Architecture:** Transfer Learning with MobileNetV2

MobileNetV2 was pre-trained on ImageNet (1.4M images, 1000 classes). We replace its classification head with our own binary head and use the pre-trained feature extractor as a frozen base.

**Why MobileNetV2?**
- Lightweight (~14MB) — fast inference, suitable for Render free tier
- Pre-trained feature extractors generalize well to medical images
- Inverted residuals + linear bottlenecks prevent gradient vanishing

**Custom head:**
```
MobileNetV2 (frozen) → GlobalAveragePooling2D → Dense(128, relu)
→ Dropout(0.3) → Dense(64, relu) → Dropout(0.2) → Dense(2, softmax)
```

**Optimization techniques used:**
- Dropout layers (regularization against overfitting)
- Adam optimizer with learning rate scheduling
- Early stopping (stops when val_loss stops improving)
- ReduceLROnPlateau (halves LR when stuck)
- ModelCheckpoint (saves best model automatically)

In [ ]:
# ── Build MobileNetV2 transfer learning model ─────────────────────────────────

INPUT_SHAPE = (IMG_SIZE, IMG_SIZE, 3)
NUM_CLASSES = 2
LEARNING_RATE_PHASE1 = 1e-4

def build_model(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES, lr=LEARNING_RATE_PHASE1):
    """
    Build and compile a MobileNetV2 transfer learning model.
    
    Phase 1: Base is FROZEN — only the custom head is trained.
    This is faster and prevents destroying pre-trained features early on.
    """
    # ---- Base model: MobileNetV2 pre-trained on ImageNet ----
    base_model = MobileNetV2(
        weights='imagenet',         # start from ImageNet weights
        include_top=False,          # remove the classification head
        input_shape=input_shape,
    )
    base_model.trainable = False    # FREEZE base layers in Phase 1

    # ---- Custom classification head ----
    inputs  = keras.Input(shape=input_shape, name='input_image')
    x       = base_model(inputs, training=False)              # feature extraction
    x       = layers.GlobalAveragePooling2D(name='gap')(x)    # reduce spatial dims
    x       = layers.Dense(128, activation='relu', name='dense_128')(x)
    x       = layers.Dropout(0.3, name='dropout_0.3')(x)      # regularization
    x       = layers.Dense(64,  activation='relu', name='dense_64')(x)
    x       = layers.Dropout(0.2, name='dropout_0.2')(x)      # regularization
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

    model = keras.Model(inputs, outputs, name='chest_xray_mobilenetv2')

    # ---- Compile with 4 metrics ----
    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=[
            'accuracy',
            Precision(name='precision'),
            Recall(name='recall'),
            AUC(name='auc'),
        ],
    )
    return model


model = build_model()
model.summary()
print(f"\nTotal parameters: {model.count_params():,}")
print(f"Trainable parameters: {sum(tf.size(w).numpy() for w in model.trainable_weights):,}")
print(f"Non-trainable parameters: {sum(tf.size(w).numpy() for w in model.non_trainable_weights):,}")

---
## Step 6 — Train Phase 1 (Frozen Base)

In Phase 1, the MobileNetV2 base is **frozen** — only the custom head layers are trained.

**Why freeze the base first?**  
The base contains 2.2M parameters pre-trained on ImageNet. If we immediately allow these to update with our small dataset, the weights will be corrupted by random head gradients — a problem called "catastrophic forgetting". By freezing the base first, we let the head find a sensible starting point before the base adapts.

**Callbacks used:**
- `EarlyStopping`: stops training if val_loss doesn't improve for 3 epochs
- `ReduceLROnPlateau`: halves the learning rate when val_loss plateaus (2 epochs patience)
- `ModelCheckpoint`: saves the best model weights automatically

In [ ]:
# ── Training Phase 1: Custom head only (frozen base) ─────────────────────────

EPOCHS_PHASE1 = 10

# Callbacks
callbacks_phase1 = [
    EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    ),
    ModelCheckpoint(
        filepath='best_model_phase1.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1,
    ),
]

print("=" * 60)
print("PHASE 1 TRAINING — Custom head only (base frozen)")
print(f"Epochs: {EPOCHS_PHASE1}, Batch size: {BATCH_SIZE}")
print("=" * 60)

history_phase1 = model.fit(
    train_ds,
    epochs=EPOCHS_PHASE1,
    validation_data=val_ds,
    callbacks=callbacks_phase1,
    verbose=1,
)

print(f"\nPhase 1 complete. Best val_accuracy: {max(history_phase1.history['val_accuracy']):.4f}")

---
## Step 7 — Fine-Tune Phase 2 (Unfreeze Top 20 Layers)

Now we unfreeze the **top 20 layers** of the MobileNetV2 base so they can adapt to the chest X-ray domain. The lower layers (which detect edges and textures) remain frozen — these generic features transfer well. Only the high-level feature detectors need domain adaptation.

**Key change:** We use a **10x lower learning rate** (1e-5 instead of 1e-4) to prevent destroying the carefully learned base weights with large gradient updates.

In [ ]:
# ── Training Phase 2: Fine-tune top 20 base layers ───────────────────────────

EPOCHS_PHASE2   = 10
LEARNING_RATE_PHASE2 = 1e-5  # 10x lower than Phase 1
NUM_UNFREEZE    = 20

# Locate the MobileNetV2 base sub-model
base_model = None
for layer in model.layers:
    if isinstance(layer, keras.Model):
        base_model = layer
        break

# Unfreeze top NUM_UNFREEZE layers, keep the rest frozen
base_model.trainable = True
freeze_until = max(0, len(base_model.layers) - NUM_UNFREEZE)
for layer in base_model.layers[:freeze_until]:
    layer.trainable = False

total_trainable = sum(tf.size(w).numpy() for w in model.trainable_weights)
print(f"Unfroze top {NUM_UNFREEZE} base layers.")
print(f"Total trainable parameters now: {total_trainable:,}")

# Recompile with lower learning rate
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE_PHASE2),
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        Precision(name='precision'),
        Recall(name='recall'),
        AUC(name='auc'),
    ],
)

callbacks_phase2 = [
    EarlyStopping(
        monitor='val_loss',
        patience=4,
        restore_best_weights=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-8,
        verbose=1,
    ),
    ModelCheckpoint(
        filepath='chest_xray_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1,
    ),
]

print("\n" + "=" * 60)
print("PHASE 2 TRAINING — Fine-tuning top 20 base layers")
print(f"Epochs: {EPOCHS_PHASE2}, LR: {LEARNING_RATE_PHASE2}")
print("=" * 60)

history_phase2 = model.fit(
    train_ds,
    epochs=EPOCHS_PHASE2,
    validation_data=val_ds,
    callbacks=callbacks_phase2,
    verbose=1,
)

print(f"\nPhase 2 complete. Best val_accuracy: {max(history_phase2.history['val_accuracy']):.4f}")

---
## Step 8 — Model Evaluation

We evaluate the model using **6 metrics** to get a complete picture of performance:

| Metric | What it measures |
|--------|------------------|
| **Accuracy** | % of all predictions that are correct |
| **Loss** | Cross-entropy loss on test set |
| **Precision** | Of all predicted PNEUMONIA, how many are truly PNEUMONIA? |
| **Recall** | Of all true PNEUMONIA cases, how many did we catch? |
| **F1 Score** | Harmonic mean of Precision and Recall |
| **ROC-AUC** | Area under the ROC curve — model's discrimination ability |

In a medical context, **Recall (sensitivity)** is most important — missing a pneumonia case is more dangerous than a false alarm.

In [ ]:
# ── Step 8a: Evaluate on test set ────────────────────────────────────────────

print("Evaluating on test set...")
test_results = model.evaluate(test_ds, verbose=1)
metric_names = model.metrics_names

print("\n" + "=" * 50)
print("TEST SET RESULTS (from model.evaluate)")
print("=" * 50)
for name, val in zip(metric_names, test_results):
    print(f"  {name:12s}: {val:.4f}")

In [ ]:
# ── Step 8b: Collect predictions for sklearn metrics ─────────────────────────

print("Collecting predictions on test set for detailed analysis...")

y_true  = []  # true labels (integers)
y_pred  = []  # predicted labels (integers)
y_probs = []  # softmax probability for PNEUMONIA class

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    # labels are one-hot: argmax gives integer label
    y_true.extend(np.argmax(labels.numpy(), axis=1).tolist())
    y_pred.extend(np.argmax(preds, axis=1).tolist())
    y_probs.extend(preds[:, 1].tolist())  # P(PNEUMONIA)

y_true  = np.array(y_true)
y_pred  = np.array(y_pred)
y_probs = np.array(y_probs)

print(f"Total test samples: {len(y_true)}")
print(f"  NORMAL (0):    {(y_true==0).sum()}")
print(f"  PNEUMONIA (1): {(y_true==1).sum()}")

In [ ]:
# ── Step 8c: Compute all 6 metrics ───────────────────────────────────────────

from sklearn.metrics import accuracy_score

accuracy   = accuracy_score(y_true, y_pred)
precision  = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
recall     = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
f1         = f1_score(y_true, y_pred, pos_label=1, zero_division=0)

fpr, tpr, _ = roc_curve(y_true, y_probs)
roc_auc     = sklearn_auc(fpr, tpr)

# Loss is from model.evaluate above
loss_val = test_results[0]

print("=" * 55)
print("FULL EVALUATION METRICS — Chest X-Ray Classifier")
print("=" * 55)
print(f"  Accuracy  : {accuracy:.4f}  ({accuracy:.2%})")
print(f"  Loss      : {loss_val:.4f}")
print(f"  Precision : {precision:.4f}  ({precision:.2%})")
print(f"  Recall    : {recall:.4f}  ({recall:.2%})")
print(f"  F1 Score  : {f1:.4f}  ({f1:.2%})")
print(f"  ROC-AUC   : {roc_auc:.4f}")
print("=" * 55)

print("\nFull Classification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Save metrics to JSON (matches what the API expects)
metrics_dict = {
    "accuracy":  round(accuracy, 6),
    "loss":      round(loss_val, 6),
    "precision": round(precision, 6),
    "recall":    round(recall, 6),
    "f1":        round(f1, 6),
    "roc_auc":   round(roc_auc, 6),
}

with open('metrics.json', 'w') as f:
    json.dump(metrics_dict, f, indent=2)

print("\nMetrics saved to metrics.json")

In [ ]:
# ── Step 8d: Confusion Matrix ─────────────────────────────────────────────────

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Confusion matrix heatmap
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    ax=axes[0],
    linewidths=1,
    linecolor='white',
    annot_kws={'size': 16, 'weight': 'bold'},
)
axes[0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_ylabel('True Label', fontsize=12)
axes[0].set_xlabel('Predicted Label', fontsize=12)

# ROC Curve
axes[1].plot(fpr, tpr, color='#5b8dee', lw=2.5, label=f'ROC Curve (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier (AUC = 0.50)')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#5b8dee')
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.02])
axes[1].set_title('ROC Curve', fontsize=14, fontweight='bold')
axes[1].set_xlabel('False Positive Rate', fontsize=12)
axes[1].set_ylabel('True Positive Rate (Recall)', fontsize=12)
axes[1].legend(loc='lower right', fontsize=11)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)
axes[1].set_aspect('equal')

plt.tight_layout()
plt.savefig('confusion_matrix_roc.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Confusion Matrix Analysis:")
print(f"  True Negatives  (NORMAL   → NORMAL):    {tn}")
print(f"  False Positives (NORMAL   → PNEUMONIA): {fp}  ← over-diagnosis")
print(f"  False Negatives (PNEUMONIA → NORMAL):  {fn}  ← missed cases (dangerous!)")
print(f"  True Positives  (PNEUMONIA → PNEUMONIA): {tp}")
print(f"\n  Sensitivity (Recall): {tp/(tp+fn):.2%}")
print(f"  Specificity:          {tn/(tn+fp):.2%}")

In [ ]:
# ── Step 8e: Learning Curves ──────────────────────────────────────────────────
# Merge Phase 1 and Phase 2 histories for a complete view

def merge_histories(h1, h2):
    """Concatenate two Keras history objects."""
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history.get(key, [])
    return merged

full_history = merge_histories(history_phase1, history_phase2)
phase1_len   = len(history_phase1.history['accuracy'])
total_epochs = len(full_history['accuracy'])
epoch_range  = range(1, total_epochs + 1)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Training History — Phase 1 (Head Only) + Phase 2 (Fine-Tuning)', fontsize=14, fontweight='bold')

# Accuracy
axes[0,0].plot(epoch_range, full_history['accuracy'],     label='Train Accuracy', color='#28a745')
axes[0,0].plot(epoch_range, full_history['val_accuracy'], label='Val Accuracy',   color='#dc3545')
axes[0,0].axvline(phase1_len, color='gray', linestyle='--', alpha=0.7, label='Phase boundary')
axes[0,0].set_title('Accuracy')
axes[0,0].set_xlabel('Epoch')
axes[0,0].set_ylabel('Accuracy')
axes[0,0].legend()
axes[0,0].set_ylim([0, 1.05])

# Loss
axes[0,1].plot(epoch_range, full_history['loss'],     label='Train Loss', color='#28a745')
axes[0,1].plot(epoch_range, full_history['val_loss'], label='Val Loss',   color='#dc3545')
axes[0,1].axvline(phase1_len, color='gray', linestyle='--', alpha=0.7, label='Phase boundary')
axes[0,1].set_title('Loss')
axes[0,1].set_xlabel('Epoch')
axes[0,1].set_ylabel('Loss')
axes[0,1].legend()

# Precision
axes[1,0].plot(epoch_range, full_history.get('precision', [0]*total_epochs),     label='Train Precision', color='#5b8dee')
axes[1,0].plot(epoch_range, full_history.get('val_precision', [0]*total_epochs), label='Val Precision',   color='#f39c12')
axes[1,0].axvline(phase1_len, color='gray', linestyle='--', alpha=0.7)
axes[1,0].set_title('Precision')
axes[1,0].set_xlabel('Epoch')
axes[1,0].set_ylabel('Precision')
axes[1,0].legend()
axes[1,0].set_ylim([0, 1.05])

# Recall
axes[1,1].plot(epoch_range, full_history.get('recall', [0]*total_epochs),     label='Train Recall', color='#5b8dee')
axes[1,1].plot(epoch_range, full_history.get('val_recall', [0]*total_epochs), label='Val Recall',   color='#f39c12')
axes[1,1].axvline(phase1_len, color='gray', linestyle='--', alpha=0.7)
axes[1,1].set_title('Recall')
axes[1,1].set_xlabel('Epoch')
axes[1,1].set_ylabel('Recall')
axes[1,1].legend()
axes[1,1].set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig('learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Learning Curve Interpretation:")
print("Phase 1 (before dotted line): Only the head trains — fast convergence, may plateau.")
print("Phase 2 (after dotted line): Top base layers unfreeze — accuracy jumps further.")
print("If train acc >> val acc, the model is overfitting → need more Dropout or augmentation.")
print("If both curves converge and stabilise, the model has generalised well.")

---
## Step 9 — Save & Download the Model

We save the final model in `.h5` (HDF5) format — widely supported, works with the FastAPI backend.

After downloading, place the file at `models/chest_xray_model.h5` in the GitHub repo.

In [ ]:
# ── Save the trained model ────────────────────────────────────────────────────

MODEL_SAVE_PATH = 'chest_xray_model.h5'
model.save(MODEL_SAVE_PATH)
print(f"Model saved to {MODEL_SAVE_PATH}")

# Check file size
size_mb = os.path.getsize(MODEL_SAVE_PATH) / (1024 * 1024)
print(f"Model file size: {size_mb:.1f} MB")

In [ ]:
# ── Download from Google Colab ────────────────────────────────────────────────
# This triggers a file download in the browser.
# Save the downloaded chest_xray_model.h5 to your project's models/ folder.

try:
    from google.colab import files
    files.download('chest_xray_model.h5')
    files.download('metrics.json')      # also download metrics for the API
    print("✅ Download triggered! Save chest_xray_model.h5 to models/ in your repo.")
except ImportError:
    print("Not running in Colab — files are saved locally.")
    print(f"Model path: {os.path.abspath('chest_xray_model.h5')}")
    print(f"Metrics path: {os.path.abspath('metrics.json')}")

---
## Step 10 — Prediction Demo

Demonstrate single-image inference — the same logic used by the FastAPI `/predict` endpoint.

In [ ]:
# ── Prediction demo: a few test images ───────────────────────────────────────

CLASS_NAMES = ['NORMAL', 'PNEUMONIA']

def predict_from_array(model, img_array, true_label=None):
    """
    Run inference on a single (1, 224, 224, 3) float32 numpy array.
    Returns a dict with class, confidence, and probabilities.
    """
    preds = model.predict(img_array, verbose=0)[0]  # shape: (2,)
    pred_idx   = int(np.argmax(preds))
    confidence = float(preds[pred_idx])
    result = {
        'class':         CLASS_NAMES[pred_idx],
        'confidence':    round(confidence, 4),
        'probabilities': {
            'NORMAL':    round(float(preds[0]), 4),
            'PNEUMONIA': round(float(preds[1]), 4),
        },
    }
    if true_label is not None:
        result['true_label'] = CLASS_NAMES[true_label]
        result['correct']    = (pred_idx == true_label)
    return result


# Show 8 test predictions
n_demo = 8
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle('Sample Test Predictions', fontsize=15, fontweight='bold')

sample_count = 0
for images, labels in test_ds:
    for i in range(len(images)):
        if sample_count >= n_demo:
            break
        img_np   = images[i].numpy()     # (224, 224, 3)
        true_lbl = int(np.argmax(labels[i].numpy()))
        img_batch = np.expand_dims(img_np, axis=0)  # (1, 224, 224, 3)

        result = predict_from_array(model, img_batch, true_label=true_lbl)

        row, col = divmod(sample_count, 4)
        ax = axes[row][col]
        ax.imshow(img_np, cmap='bone')
        color = '#28a745' if result['correct'] else '#dc3545'
        ax.set_title(
            f"Pred: {result['class']} ({result['confidence']:.1%})\nTrue: {result['true_label']}",
            color=color, fontsize=10, fontweight='bold'
        )
        ax.axis('off')
        sample_count += 1

    if sample_count >= n_demo:
        break

plt.tight_layout()
plt.savefig('prediction_demo.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Prediction demo complete!")
print("Green title = correct prediction, Red title = incorrect prediction.")

---
## Summary

| Step | Status |
|------|--------|
| Data loading (tensorflow_datasets) | ✅ |
| Preprocessing (resize, normalize, augment) | ✅ |
| EDA: 3 feature interpretations | ✅ |
| MobileNetV2 transfer learning model | ✅ |
| Phase 1 training (frozen base + early stopping) | ✅ |
| Phase 2 fine-tuning (top 20 layers unfrozen) | ✅ |
| 6 evaluation metrics (Accuracy, Loss, F1, Precision, Recall, ROC-AUC) | ✅ |
| Confusion matrix + ROC curve | ✅ |
| Learning curves (all 4 phases) | ✅ |
| Model saved as chest_xray_model.h5 | ✅ |
| Metrics saved as metrics.json | ✅ |
| Prediction demo | ✅ |

**Next steps:**
1. Download `chest_xray_model.h5` → place in `models/` folder
2. Download `metrics.json` → place in `models/` folder
3. Run `uvicorn api.main:app --reload` to start the API
4. Run `streamlit run ui/app.py` to start the dashboard
5. Deploy to Render following the README instructions
